## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

Async LangGraph Agent with Groq + Tavily + Phoenix OpenTelemetry Tracing
==========================================================================

This lab demonstrates:
1. LangGraph ReAct agent (modern replacement for deprecated LangChain agents)
2. Groq LLM (fast inference via OpenAI-compatible endpoint)
3. Tavily Search via langchain-tavily (modern package)
4. OpenTelemetry tracing with Phoenix visualization
5. Async concurrent execution


In [48]:
import os
import asyncio
from typing import Any
from dotenv import load_dotenv

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, trace
from agents import ModelSettings


# ============================================================================
# 1. DEPENDENCIES
# ============================================================================
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch  # Modern package (correct import)
from langgraph.prebuilt import create_react_agent  # Modern agent
from langchain_core.tools import tool

# OpenTelemetry + Phoenix tracing (matching lab1test.ipynb pattern)
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry import trace as otel_trace
# from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor
from openinference.instrumentation.langchain import LangChainInstrumentor


In [49]:
# ============================================================================
# 2. ENVIRONMENT SETUP
# ============================================================================

load_dotenv(override=True)

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
TAVILY_API_KEY = os.environ.get("TAVILY_API_KEY")
PHOENIX_OTLP_GRPC = os.environ.get("PHOENIX_OTLP_GRPC", "192.168.0.111:30317")
PHOENIX_UI = os.environ.get("PHOENIX_UI", "http://192.168.0.111:30606")

print(f"✓ GROQ_API_KEY loaded")
print(f"✓ TAVILY_API_KEY loaded")


# Configure tracing — sends spans to Phoenix over gRPC (plain, no TLS)

provider = TracerProvider()
provider.add_span_processor(
    BatchSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX_OTLP_GRPC, insecure=True))
)
otel_trace.set_tracer_provider(provider)
LangChainInstrumentor().instrument(tracer_provider=provider)

print("Trace setup done")

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


✓ GROQ_API_KEY loaded
✓ TAVILY_API_KEY loaded
Trace setup done


In [50]:
tavily_search = TavilySearch(
    max_results=5
)

tools = [tavily_search]

model = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    openai_api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    temperature=0.7,
)


In [51]:
INSTRUCTIONS = """You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself. \
CRITICAL RULES: \
1. You MUST use the tavily_search tool for EVERY query \
2. Do NOT provide any response without first calling tavily_search

Your workflow is ALWAYS:
Step 1: Call tavily_search with the search term
Step 2: send the raw results from tavily_search to the user with a message like "Search results: {{results}}"
    """

# Create LangGraph ReAct agent - CORRECTED

search_agent = create_react_agent(
    name="Search agent",
    model=model,
    tools=tools,
    prompt=INSTRUCTIONS,
)

In [ ]:
# Run the agent (LangGraph uses invoke, not Runner.run)
message = "Latest AI Agent frameworks in 2026"
result = search_agent.invoke({"messages": [("user", message)]})

final_message = result["messages"][-1].content
display(Markdown(final_message))

In [52]:
def run_with_trace(query, trace_name="Search"):
    with otel_trace.get_tracer(__name__).start_as_current_span(trace_name):
        result = search_agent.invoke({"messages": [("user", query)]})
        return result["messages"][-1].content

message = "Latest AI Agent frameworks"
response = run_with_trace(message)
display(Markdown(response))

Search results: {"query": "Latest AI Agent frameworks", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://vellum.ai/blog/top-ai-agent-frameworks-for-developers", "title": "The Top 11 AI Agent Frameworks For Developers In September 2026", "content": "__wf_reserved_inherit\n\nQuick overview: Vellum AI is a production-grade AI agent framework built for developers who need reliability, observability, and tight control. It gives engineering teams a unified environment to build, test, and deploy agents using a powerful TypeScript and Python SDK, a visual editor for rapid iteration, and a natural-language Agent Builder for fast scaffolding. With built-in evaluations, versioning, and end-to-end observability, developers can debug, compare, and validate agent behavior with confidence. Vellum also supports flexible deployment across cloud, VPC, hybrid, or on-prem environments, making it easy for engineering, product, and compliance to collaborate on agents that are truly production-ready. [...] ## Quick recommendations\n\nNeed enterprise controls, audit trails, and fast iteration across teams: Choose Vellum . Building deep custom logic with multiple models and tools: Choose LangChain . Prototyping GPT assistants fast with built-in guardrails: Choose OpenAI Agents SDK . Researching multi-agent self-reflection loops: Choose AutoGen . Designing role-based teams visually: Choose CrewAI . Connecting apps and AI in low-code workflows: Choose n8n or Zapier . Want OSS visual builder with templates: Choose Dify .\n\n# Why choose Vellum\n\nVellum is the AI agent platform that lets non-technical teammates and developers co-build reliable, testable, observable AI agent that scale. If you care about moving from pilots to production without slowing collaboration, Vellum is the right choice. [...] ★★★★☆ CrewAI Enterprise only Visual builder; role-based team of agents Collaborative agent teams ★★★★☆ n8n Free; $20/mo cloud Low-code canvas; hundreds of integrations; self-host AI and SaaS workflow automation ★★★★☆ Zapier Free; from $19.99/mo No-code; massive app ecosystem Non-technical automation and AI ★★★★☆ Lindy AI From $29/mo Assistant templates; customizable flows Personal and business assistants ★★★☆☆ Gumloop Free; from $37/mo Drag-and-drop; prototyping templates Rapid LLM agent prototyping ★★★☆☆ Stack AI Free; Enterprise Low-code editor; API integrations AI workflow automation ★★★☆☆ Dify Free (self-hosted); paid cloud Visual builder; OSS flexibility; templates Open-source agent orchestration ★★★☆☆", "score": 0.77179235, "raw_content": null}, {"url": "https://www.linkedin.com/pulse/ai-agent-frameworks-2026-how-choose-build-scale-agentic-systems-ew8qf", "title": "AI Agent Frameworks 2026: How to Choose, Build & Scale Agentic ...", "content": "Scoring Model: Frameworks are scored using weighted criteria tailored to client constraints, enabling informed architectural decisions rather than misleading “best framework” claims.\n\n### Deep Dive: Leading AI Agent Frameworks (2026)\n\n### 1. LangGraph\n\nLangGraph is designed for teams that require structure, predictability, and controllability in the development of agentic systems. It targets deterministic workflows, where each step of an agent’s execution takes a well-defined course. This makes LangGraph highly useful in a production setting where predictability and system stability are more important than creative freedom. [...] AI agents should not be viewed as chatbots. While artificial intelligence (AI) chatbots respond to their users' input, an AI agent can pursue goals for the user's benefit. They make decisions about action plans, tools to use, and when to reach out for assistance. The shift from engaging with static language models to creating dynamic patterns of behaviour aimed at achieving goals represents a profound transition in the use of applied AI.\n\nThe years 2025-26 represent the true development of an AI agent framework that will meet the needs of business customers. The capabilities of AI agents are maturing, and the supporting ecosystem of AI tools is expanding. Companies must now deliver measurable value from deployed AI systems, not simply a fascinating demonstration of technology. [...] AI Agent Frameworks 2026: How to Choose, Build & Scale Agentic Systems\n\n# AI Agent Frameworks 2026: How to Choose, Build & Scale Agentic Systems\n\nDextra Labs\n\n### Dextra Labs\n\n#### De-Risking Enterprise AI and Tech DD. Designing Production-Grade Systems.\n\nOver the last several years, AI has matured far beyond simple conversational interfaces or experimental demonstrations. Enterprises are now developing AI agents capable of reasoning, planning for the future, taking actions and adapting over time. These systems are no longer just reactive. They are becoming true autonomous partners by providing direct integration into live corporate business processes.", "score": 0.7382357, "raw_content": null}, {"url": "https://www.workday.com/en-us/perspectives/ai/top-ai-agent-frameworks.html", "title": "The 5 Best AI Agent Frameworks for Scalable Workflows | Workday US", "content": "Understanding each framework’s design and the real-world examples of AI agents they power helps teams match their agent use cases to the right foundation and architecture. Here are five of the top AI agent frameworks available to businesses in 2026.\n\n### 1. LangChain + LangGraph\n\nLangChain is an open-source framework for building AI applications with large language models, and LangGraph extends it with a graph-based runtime for long-running, stateful workflows and agents.\n\nKey capabilities:\n\nBest suited for: Teams that want fine-grained control over agent workflows, especially multi-step or multi-agent applications that benefit from explicit graph structure and state management.\n\n### 2. AutoGen [...] Understanding each framework’s design and the real-world examples of AI agents they power helps teams match their agent use cases to the right foundation and architecture. Here are five of the top AI agent frameworks available to businesses in 2026.\n\n### 1. LangChain + LangGraph\n\nLangChain is an open-source framework for building AI applications with large language models, and LangGraph extends it with a graph-based runtime for long-running, stateful workflows and agents.\n\nKey capabilities: [...] ### 5. CrewAI\n\nCrewAI is an open-source Python framework for building and orchestrating multi-agent \"crews,” or groups of specialized agents that collaborate to complete tasks.\n\nKey capabilities:\n\n Role-based agents and crews: Lets developers define agents with specific roles and skills, then organize them into coordinated crews for end-to-end workflows\n Built-in guardrails and memory: Includes mechanisms for memory management, knowledge, and guardrails to help keep multi-agent interactions on track\n Developer and UI tooling: Offers both a code-first experience and visual tools for designing, testing, and deploying multi-agent workflows", "score": 0.71496695, "raw_content": null}, {"url": "https://www.ibm.com/think/insights/top-ai-agent-frameworks", "title": "AI Agent Frameworks: Choosing the Right Foundation for Your ... - IBM", "content": "### Data Insights with LangGraph and watsonx.ai\n\nCan an AI agent take our natural language query and do the processing for us to give us that meaningful output? We use several pieces of open source technology and the power of watsonx.ai to put this to the test.\n\n## Popular AI agent frameworks\n\nAgentic AI is still in its early stages. As the technology behind AI agents evolves, so too will the frameworks underlying them. Here are some currently popular AI agent frameworks:\n\n### AutoGen\n\nAutoGen is an open-source framework from Microsoft for creating multiagent AI applications to perform complex tasks. Its architecture consists of 3 layers: [...] # AI agent frameworks: Choosing the right foundation for your business\n\n## Authors\n\nStaff Writer\n\nIBM Think\n\nStaff Editor, AI Models\n\nIBM Think\n\nFrom a single artificial intelligence (AI) agent that monitors and flags fraudulent transactions for financial institutions to a multiagent system for supply chain management that tracks inventory levels and forecasts demand, agentic AI can be a boon for businesses. So how can enterprises get started with AI agents? This is where AI agent frameworks come in.\n\n## The latest AI trends, brought to you by experts\n\nGet curated insights on the most important—and intriguing—AI news. Subscribe to our twice-weekly Think Newsletter. See the IBM Privacy Statement.\n\n## Thank you!\n\nYou are subscribed. [...] ### LlamaIndex\n\nLlamaIndex is an open-source data orchestration framework for building generative AI (gen AI) and agentic AI solutions. It offers prepackaged agents and tools and recently introduced workflows, a mechanism for developing multiagent systems.\n\nHere are the main elements that make up a workflow in LlamaIndex:\n\nThis event-driven architecture enables workflow steps to be accomplished asynchronously. This means that, unlike a graph architecture, the paths between steps don’t need to be defined, resulting in more flexible transitions between agent actions.\n\nAs such, LlamaIndex workflows are well-suited for more dynamic AI agent applications that need to loop back often to previous steps or branch to several steps.\n\nLlamaIndex is available to access on GitHub.", "score": 0.6947383, "raw_content": null}, {"url": "https://www.salesforce.com/agentforce/ai-agents/ai-agent-frameworks/", "title": "AI Agent Frameworks: A Practical Guide (2026) - Salesforce", "content": "## New to Salesforce\n\nSalesforce is the #1 AI CRM, with AI agents, data, and CRM apps on a single, unified platform.\n\nLearn all about what CRM is, what it does, and how it can improve customer relationships.\n\nSalesforce’s category-leading apps for sales, service, marketing, commerce, IT, and more.\n\nLearn more about AI agents and how they can help your business.\n\nIt’s always-on digital labor augmenting every employee, department, and business process to improve customer experiences.\n\n## Blogs\n\nDiscover tips and insights from experts to supercharge your digital journey.\n\nGet the latest on AI, data, digital labor, and more.\n\nKeep up with Salesforce’s AI research team and stay on the cutting edge of technology.\n\nGet the latest tooling innovation news, insights, and inspiration. [...] ## Agentforce World Tours\n\nJoin us in cities around the world to experience the latest Salesforce innovations in apps, data, and agents. Connect with leaders and experts from your industry at an Agentforce World Tour near you.\n\nBrowse current Agentforce World Tour events, or sign up today to be the first to know when one is coming to a city near you\n\nExplore 140+ expert-led sessions, demos, and hands-on trainings featuring Dreamforce’s boldest launches. All in one day, all for free.\n\nExperience the Dreamforce innovations that power Agentic Enterprises with expert-led sessions, demos, and hands-on trainings. All in one day, all for free.\n\n## Salesforce+\n\nFuel your professional growth and transform your company with our free live streaming and on-demand events. [...] mobile menu open\nmobile menu close\nSalesforce Home\nSalesforce Home\nSalesforce Home\n\n## Products\n\n## Meet Agentforce 360\n\nBring people, apps, data, and AI agents together on one platform to increase productivity, improve operations, and build stronger customer relationships.\n\n## Agentforce\n\nHumans with Agents drive customer success together.\n\n## Sales\n\nBoost pipeline, win rate, and revenue with Sales Cloud.\n\n## Service\n\nCut service costs with humans & AI agents on one platform.\n\n## Marketing\n\nPersonalize every moment of engagement across the customer lifecycle with AI agents, actionable data, and workflows.\n\n## Commerce\n\nIncrease revenue and deliver consistent customer experiences across online, in-store, and mobile channels with Commerce Cloud.\n\n## Analytics", "score": 0.5596998, "raw_content": null}], "response_time": 3.41, "request_id": "6a1205d2-745e-45c5-8df6-36a9a49650ac"}

### As always, take a look at the trace

https://platform.openai.com/traces

### We will now use Structured Outputs, and include a description of the fields

In [33]:
# See note above about cost of WebSearchTool

HOW_MANY_SEARCHES = 3

INSTRUCTIONS = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."

# Use Pydantic to define the Schema of our response - this is known as "Structured Outputs"
# With massive thanks to student Wes C. for discovering and fixing a nasty bug with this!

class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")

    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=WebSearchPlan,
)

In [ ]:

message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(planner_agent, message)
    print(result.final_output)

In [ ]:
@function_tool
def send_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("ed@edwarddonner.com") # Change this to your verified email
    to_email = To("ed.donner@gmail.com") # Change this to your email
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return "success"

In [ ]:
send_email

In [37]:
INSTRUCTIONS = """You are able to send a nicely formatted HTML email based on a detailed report.
You will be provided with a detailed report. You should use your tool to send one email, providing the 
report converted into clean, well presented HTML with an appropriate subject line."""

email_agent = Agent(
    name="Email agent",
    instructions=INSTRUCTIONS,
    tools=[send_email],
    model="gpt-4o-mini",
)



In [38]:
INSTRUCTIONS = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")

    markdown_report: str = Field(description="The final report")

    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    output_type=ReportData,
)

### The next 3 functions will plan and execute the search, using planner_agent and search_agent

In [39]:
async def plan_searches(query: str):
    """ Use the planner_agent to plan which searches to run for the query """
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    print(f"Will perform {len(result.final_output.searches)} searches")
    return result.final_output

async def perform_searches(search_plan: WebSearchPlan):
    """ Call search() for each item in the search plan """
    print("Searching...")
    tasks = [asyncio.create_task(search(item)) for item in search_plan.searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: WebSearchItem):
    """ Use the search agent to run a web search for each item in the search plan """
    input = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input)
    return result.final_output

### The next 2 functions write a report and email it

In [40]:
async def write_report(query: str, search_results: list[str]):
    """ Use the writer agent to write a report based on the search results"""
    print("Thinking about report...")
    input = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input)
    print("Finished writing report")
    return result.final_output

async def send_email(report: ReportData):
    """ Use the email agent to send an email with the report """
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return report

### Showtime!

In [ ]:
query ="Latest AI Agent frameworks in 2025"

with trace("Research trace"):
    print("Starting research...")
    search_plan = await plan_searches(query)
    search_results = await perform_searches(search_plan)
    report = await write_report(query, search_results)
    await send_email(report)  
    print("Hooray!")




### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>